In [6]:
%pip install -q lightgbm scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd

TRAIN_DATA = pd.read_csv('train-data.csv', index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA = pd.read_csv('test-data.csv', index_col='id')

In [8]:
import numpy as np
import pandas as pd

def preprocess(df):
    df = df.copy()

    # === DROP: near-zero signal or too high cardinality ===
    drop_cols = [
        'first_name', 'last_name',
        'insitute_name', 'institute_location',
    ]
    df = df.drop(columns=drop_cols)

    # === test_1~5 and treatment_consent: only 1 unique value
    # The NaN itself may carry signal — encode as "was it missing?"
    for col in ['test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'treatment_consent']:
        df[col + '_missing'] = df[col].isna().astype(int)
    df = df.drop(columns=['test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'treatment_consent'])

    # === Binary Y/N -> 1/0, NaN stays NaN (LightGBM handles it) ===
    binary_yn = [
        'mother_defect', 'father_defect', 'maternal_defect', 'paternal_defect',
        'alive', 'folic_acid', 'maternal_illness', 'infertility_treatment',
        'problem_previous_pregnancies',
        'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5'
    ]
    for col in binary_yn:
        df[col] = df[col].map({'Y': 1, 'N': 0})

    df['respiration'] = df['respiration'].map({'A': 1, 'N': 0})
    df['heart_rate']  = df['heart_rate'].map({'A': 1, 'N': 0})
    df['risk_level']  = df['risk_level'].map({'H': 1, 'L': 0})
    df['place_birth'] = df['place_birth'].map({'I': 1, 'H': 0})
    df['birth_defects'] = df['birth_defects'].map({'S': 1, 'M': 2})
    df['gender'] = df['gender'].map({'M': 0, 'F': 1, 'A': 2})
    df['alive']  = df['alive'].map({'Y': 1, 'N': 0})
    df['autopsy'] = df['autopsy'].map({'Y': 1, 'N': 0})

    # Y=1, N=0, NR=2 (recorded-but-unknown is different from N)
    for col in ['birth_asphyxia', 'radiation_exposure', 'substance_abuse']:
        df[col] = df[col].map({'Y': 1, 'N': 0, 'NR': 2})

    # blood_test ordinal
    df['blood_test'] = df['blood_test'].map({'N': 0, 'I': 1, 'S': 2, 'A': 3})

    # === ENGINEERED FEATURES (the real signal!) ===
    defect_cols  = ['mother_defect', 'father_defect', 'maternal_defect', 'paternal_defect']
    symptom_cols = ['symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5']

    df['defect_sum']  = df[defect_cols].sum(axis=1)
    df['symptom_sum'] = df[symptom_cols].sum(axis=1)

    # Interaction: defect * symptom (the combo table showed this is very powerful)
    df['defect_x_symptom'] = df['defect_sum'] * df['symptom_sum']

    # Any defect at all? Any symptom at all?
    df['any_defect']  = (df['defect_sum'] > 0).astype(int)
    df['any_symptom'] = (df['symptom_sum'] > 0).astype(int)

    # High symptom burden flag (4 or 5 symptoms = strong class 0/1/2 signal)
    df['high_symptom'] = (df['symptom_sum'] >= 4).astype(int)

    # All defects flag (4 defects = strong class 0/8 signal)
    df['all_defects'] = (df['defect_sum'] == 4).astype(int)

    # Parent age gap (sometimes meaningful in genetic conditions)
    df['parent_age_gap'] = (df['father_age'] - df['mother_age']).abs()

    return df

X_train = preprocess(TRAIN_DATA)
X_test  = preprocess(TEST_DATA)
y_train = TRAIN_LABEL['disorder'].values

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Features:", list(X_train.columns))

X_train shape: (13249, 45)
X_test shape: (8834, 45)
Features: ['age', 'gender', 'mother_defect', 'father_defect', 'maternal_defect', 'paternal_defect', 'blood_cell_count', 'mother_age', 'father_age', 'alive', 'respiration', 'heart_rate', 'risk_level', 'birth_asphyxia', 'autopsy', 'place_birth', 'folic_acid', 'maternal_illness', 'radiation_exposure', 'substance_abuse', 'infertility_treatment', 'problem_previous_pregnancies', 'abortion_cnt', 'birth_defects', 'white_blood_cell_count', 'blood_test', 'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5', 'test_1_missing', 'test_2_missing', 'test_3_missing', 'test_4_missing', 'test_5_missing', 'treatment_consent_missing', 'defect_sum', 'symptom_sum', 'defect_x_symptom', 'any_defect', 'any_symptom', 'high_symptom', 'all_defects', 'parent_age_gap']


In [9]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import numpy as np

N_SPLITS = 10
SEED = 42

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_preds = np.zeros(len(y_train))
test_preds = np.zeros((len(X_test), 10))

lgbm_params = dict(
    n_estimators=2000,
    learning_rate=0.02,       # slower learning = better generalization
    num_leaves=31,            # smaller than before = less overfit
    max_depth=6,
    subsample=0.7,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,           # stronger L2
    min_child_samples=30,     # needs more samples per leaf
    class_weight='balanced',  # critical for imbalanced classes 4 and 8
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    model = LGBMClassifier(**lgbm_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            early_stopping(100, verbose=False),
            log_evaluation(False)
        ]
    )

    val_pred = model.predict(X_val)
    score = balanced_accuracy_score(y_val, val_pred)
    fold_scores.append(score)
    print(f"Fold {fold+1}: BA = {score:.4f}  (best iter: {model.best_iteration_})")

    oof_preds[val_idx] = val_pred
    test_preds += model.predict_proba(X_test) / N_SPLITS

oof_score = balanced_accuracy_score(y_train, oof_preds)
print(f"\nOOF Balanced Accuracy: {oof_score:.4f}")
print(f"Mean Fold BA: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

Fold 1: BA = 0.3195  (best iter: 471)
Fold 2: BA = 0.2587  (best iter: 496)
Fold 3: BA = 0.3128  (best iter: 565)
Fold 4: BA = 0.2925  (best iter: 540)
Fold 5: BA = 0.3078  (best iter: 603)
Fold 6: BA = 0.2774  (best iter: 544)
Fold 7: BA = 0.2897  (best iter: 498)
Fold 8: BA = 0.2893  (best iter: 528)
Fold 9: BA = 0.3292  (best iter: 559)
Fold 10: BA = 0.2832  (best iter: 549)

OOF Balanced Accuracy: 0.2958
Mean Fold BA: 0.2960 ± 0.0202


In [10]:
final_preds = np.argmax(test_preds, axis=1)

submission = pd.DataFrame({
    'id': TEST_DATA.index,
    'disorder': final_preds
}).set_index('id')

submission.to_csv('submission-a1.csv')
print("Saved! Shape:", submission.shape)
print(submission['disorder'].value_counts().sort_index())

Saved! Shape: (8834, 1)
disorder
0     266
1    1512
2     854
3    1806
4      38
5    1385
6     892
7    1349
8      47
9     685
Name: count, dtype: int64
